#init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import  col, trim 

##get from bronze

In [0]:
df = spark.table('workspace.bronze.erp_px_cat_g1v2')

## preview df

In [0]:
df.display()

#Transformation

##trim

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##normalize Maintainance Flag to bool

In [0]:
df = df.withColumn(
    'maintainance',
    F.when(F.upper(col('MAINTENANCE'))== 'YES', F.lit(True))
     .when(F.upper(col('MAINTENANCE'))== 'NO', F.lit(False))
     .otherwise(None)
)

##renaming columns

In [0]:
rename_map = {
    'ID': 'category_id',
    'CAT': 'category',
    'SUBCAT': 'subcategory',
    'MAINTENANCE': 'maintainance_flag'
}

for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

## validate df

In [0]:
df.limit(10).display()

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.erp_product_category')

In [0]:
%sql
SELECT * FROM workspace.silver.erp_product_category
